# Batch Proposal Filtering

After running acquisition function optimization (`cleo-optimize-batch`) with
several `gamma` values, you will have a pool of candidate sequences ranked by
predicted mean activity (mu) and batch diversity. This notebook walks through
the process of filtering and selecting the final batch to test experimentally.

We cover:
1. Loading candidates from multiple gamma sweeps
2. Removing sequences that have already been tested
3. Visualizing the predicted activity distribution
4. Inspecting the optimization metrics across gamma values
5. Computing pairwise diversity for the candidate pool
6. Applying a mu threshold and re-computing diversity
7. Selecting the final batch (top-k by mu + most diverse)
8. Visualizing mutations relative to the parent
9. Exporting the batch as a FASTA file

See the [README section on batch proposals](../README.md#-proposing-batch-of-sequences-to-test-next)
for the full discussion on selection strategy.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from matplotlib.colors import ListedColormap, BoundaryNorm
from plot_utils import setup_style, CLEO_PALETTE, PRIMARY, SECONDARY, COLOR_REFERENCE

setup_style()

## Configuration

Paths and experiment-specific settings. Adjust these to match your
directory layout.

In [ ]:
connector = "___"
num_fragments = 5

# The gamma values used during acquisition function optimization
gamma_values = [0, 1, 2, 5, 10]

# Reference sequence for mutation visualization
ref_seq = (
    "MGEEEELELERPSGERTPVRRHRFPARKANNIEEAVANVERL"
    "IEEIEASGITFTATADRAVVVGWSLGVITGMIMHATGTDFITAL"
    "RKALEIGKKVKEEDPEFMERHKKIVTDGNRAEIREDIDYWIE"
    "VIEKETGHPVDRSRIFIAETVEEAVELARRAVELGHA"
    "IIVLPPYLIGTAGEAVVQVLASAGVDVLLMGGLGSGVPVTIYRA"
)

fragment_bounds = [[0, 41], [42, 85], [86, 127], [128, 164], [165, 208]]

# Well map for 384-well plates
plate_rows = list("ABCDEFGHIJKLMNOP")
plate_cols = list(range(1, 25))
wp384 = [f"{r}{c}" for r in plate_rows for c in plate_cols]

## 1. Load Optimized Candidates

Each gamma run produces a `candidates.csv` with columns including `name`,
`sequence`, `mu` (predicted mean), and `sigma` (predicted uncertainty).
We concatenate candidates from all gamma values.

In [ ]:
# Replace with the path to your acquisition function optimization outputs
acqf_output_dir = Path("../outputs/acqf_optimization/")

opt_data_list = []
for g in gamma_values:
    candidates_fp = acqf_output_dir / f"gamma_{g}" / "candidates.csv"
    if not candidates_fp.exists():
        print(f"  [skip] {candidates_fp} not found")
        continue
    df_g = pd.read_csv(candidates_fp)
    df_g = df_g.sort_values(by="mu", ascending=False).reset_index(drop=True)
    df_g["gamma"] = g
    opt_data_list.append(df_g)

if opt_data_list:
    opt_data = pd.concat(opt_data_list, axis=0).reset_index(drop=True)
    print(f"Loaded {len(opt_data)} total candidates across {len(opt_data_list)} gamma values")
else:
    print("No candidate files found — update acqf_output_dir above.")
    print("Generating synthetic example data for demonstration...")

    np.random.seed(42)
    n_demo = 500
    aa = list("ACDEFGHIKLMNPQRSTVWY")
    demo_rows = []
    for g in gamma_values:
        for _ in range(n_demo // len(gamma_values)):
            seq = list(ref_seq)
            n_mut = np.random.randint(3, 15)
            positions = np.random.choice(len(seq), size=n_mut, replace=False)
            for p in positions:
                seq[p] = np.random.choice(aa)
            demo_rows.append({
                "name": f"demo_{len(demo_rows)}",
                "sequence": "".join(seq),
                "mu": np.random.normal(3.0 + g * 0.1, 1.5),
                "sigma": abs(np.random.normal(0.5, 0.3)),
                "gamma": g,
            })
    opt_data = pd.DataFrame(demo_rows)

## 2. Remove Already-Tested Sequences

Filter out any candidate whose sequence has already been tested in previous
rounds, and also deduplicate within the candidate pool.

In [ ]:
# Load previously tested sequences (replace with your training data path)
tested_data_path = Path("../example_data/data_processing/output/260128_processed_filtered_data_round4.csv")
if tested_data_path.exists():
    tested_df = pd.read_csv(tested_data_path)
    tested_seqs = set(tested_df["sequence"].tolist())
    print(f"Loaded {len(tested_seqs)} previously tested sequences")
else:
    tested_seqs = set()
    print("No tested-sequences file found; skipping dedup against past rounds.")

# Deduplicate
seen = set(tested_seqs)
filtered_rows = []
for _, row in opt_data.iterrows():
    if row["sequence"] not in seen:
        seen.add(row["sequence"])
        filtered_rows.append(row)

filtered_data = pd.DataFrame(filtered_rows).reset_index(drop=True)
print(f"Before dedup: {len(opt_data)}  →  After: {len(filtered_data)}")

## 3. Visualize Predicted Activity Distribution

Inspect the distribution of predicted mu across gamma values to understand
how the exploration–exploitation trade-off affects the candidate pool.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=filtered_data, x="mu", hue="gamma", bins=30, alpha=0.6, palette=CLEO_PALETTE)
plt.xlabel("Predicted mu")
plt.ylabel("Count")
plt.title("Distribution of Predicted Mean Activity by Gamma")
plt.tight_layout()
plt.show()

## 4. Optimization Metrics (Optional)

If your acquisition function optimization saved per-step metrics, you can
inspect how UCB and sequence similarity evolved during optimization.

In [ ]:
metrics_list = []
for g in gamma_values:
    metrics_fp = acqf_output_dir / f"gamma_{g}" / "metrics.csv"
    if metrics_fp.exists():
        df_m = pd.read_csv(metrics_fp)
        df_m["gamma"] = g
        metrics_list.append(df_m)

if metrics_list:
    metrics_df = pd.concat(metrics_list).reset_index(drop=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.lineplot(data=metrics_df, x="step", y="ucb", hue="gamma", ax=axes[0], palette=CLEO_PALETTE)
    axes[0].set_title("UCB vs. Step")
    if "seq_similarity" in metrics_df.columns:
        sns.lineplot(data=metrics_df, x="step", y="seq_similarity", hue="gamma", ax=axes[1], palette=CLEO_PALETTE)
        axes[1].set_title("Batch Similarity vs. Step")
    plt.tight_layout()
    plt.show()
else:
    print("No per-step metrics files found; skipping optimization curve plots.")

## 5. Compute Pairwise Diversity

For each candidate, compute its average Hamming distance to all other
candidates. Sequences with high pairwise diversity cover more of the
sequence space and provide more informative training data.

**Note**: This is O(n²) — for very large candidate pools you may want to
subsample first or use a faster distance implementation.

In [ ]:
def compute_pairwise_diversity(sequences):
    """Compute mean Hamming distance of each sequence to all others."""
    n = len(sequences)
    diversity = np.zeros(n)
    for i in tqdm(range(n), desc="Pairwise diversity"):
        dists = []
        for j in range(n):
            if i != j:
                dists.append(sum(a != b for a, b in zip(sequences[i], sequences[j])))
        diversity[i] = np.mean(dists) if dists else 0
    return diversity


all_seqs = filtered_data["sequence"].tolist()
filtered_data["pairwise_diversity"] = compute_pairwise_diversity(all_seqs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(filtered_data["pairwise_diversity"], bins=30, ax=axes[0], edgecolor="white", color=PRIMARY)
axes[0].set_xlabel("Mean Pairwise Hamming Distance")
axes[0].set_title("Diversity Distribution")

sns.scatterplot(
    data=filtered_data, x="mu", y="pairwise_diversity",
    hue="gamma", alpha=0.5, ax=axes[1], palette=CLEO_PALETTE,
)
axes[1].set_xlabel("Predicted mu")
axes[1].set_ylabel("Mean Pairwise Diversity")
axes[1].set_title("Predicted Activity vs. Diversity")

plt.tight_layout()
plt.show()

## 6. Apply a Mu Threshold

Set a minimum predicted activity threshold. Look at the distribution of
predictions on your **training data** to calibrate what mu values are
meaningful — especially in early rounds, the model's absolute predictions
may not be well-calibrated, so a relative threshold (e.g. top 10% of
training predictions) is safer than an absolute one.

After thresholding, recompute diversity on the smaller pool.

In [ ]:
mu_threshold = 4.0  # adjust based on your training data distribution

thresholded = filtered_data[filtered_data["mu"] > mu_threshold].reset_index(drop=True)
print(f"Candidates above mu > {mu_threshold}: {len(thresholded)} (from {len(filtered_data)})")

if len(thresholded) > 0:
    thresholded["pairwise_diversity"] = compute_pairwise_diversity(
        thresholded["sequence"].tolist()
    )

In [ ]:
if len(thresholded) > 0:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(
        data=thresholded, x="mu", y="pairwise_diversity",
        hue="gamma", alpha=0.5, palette=CLEO_PALETTE,
    )
    plt.xlabel("Predicted mu")
    plt.ylabel("Mean Pairwise Diversity")
    plt.title(f"Mu vs. Diversity (mu > {mu_threshold})")
    plt.tight_layout()
    plt.show()

## 7. Select Final Batch

A common strategy is to take a mix of:
- **Top-k by predicted mu** — exploit the model's best guesses
- **Most diverse** — explore sequence space for better future models

The exact ratio depends on your budget and how many rounds remain.
Early rounds benefit from more exploration; later rounds can lean toward
exploitation.

In [ ]:
plate_capacity = 382  # 384 - 2 controls
top_k_mu = 10

pool = thresholded if len(thresholded) > 0 else filtered_data

# Top-k by mu
mu_sorted = pool.sort_values(by="mu", ascending=False)
selected_seqs = []
selected_rows = []

for _, row in mu_sorted.head(top_k_mu).iterrows():
    selected_seqs.append(row["sequence"])
    selected_rows.append({**row.to_dict(), "selection_tag": "top_mu"})

# Fill remaining with most diverse (that haven't already been selected)
diversity_sorted = pool.sort_values(by="pairwise_diversity", ascending=False)
for _, row in diversity_sorted.iterrows():
    if len(selected_rows) >= plate_capacity:
        break
    if row["sequence"] not in selected_seqs:
        selected_seqs.append(row["sequence"])
        selected_rows.append({**row.to_dict(), "selection_tag": "diverse"})

batch_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print(f"Final batch: {len(batch_df)} sequences")
print(f"  top_mu: {(batch_df['selection_tag'] == 'top_mu').sum()}")
print(f"  diverse: {(batch_df['selection_tag'] == 'diverse').sum()}")
print(f"  mu range: [{batch_df['mu'].min():.2f}, {batch_df['mu'].max():.2f}]")

### Fragment Coverage

Check how many unique fragments per position are represented in the
selected batch. Good fragment coverage means the batch will generate
informative interaction data.

In [ ]:
if "name" in batch_df.columns and batch_df["name"].str.contains(connector).any():
    for i in range(num_fragments):
        frags = batch_df["name"].str.split(connector).str[i]
        frags = frags.apply(lambda x: x.split(":")[0] if isinstance(x, str) else x)
        n_unique = frags.nunique()
        print(f"Fragment {i+1}: {n_unique} unique out of {len(frags)} total")
else:
    print("No fragment names available (demo data); skipping coverage analysis.")

## 8. Mutation Heatmap

Visualize where mutations occur in the selected batch relative to the
parent sequence. Fragment boundaries are shown as red dashed lines.

In [ ]:
aa_list = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i + 1 for i, aa in enumerate(aa_list)}

num_to_plot = min(len(batch_df), 384)
plot_seqs = batch_df["sequence"].head(num_to_plot).tolist()

diff_matrix = []
for seq in plot_seqs:
    row = []
    for j, aa in enumerate(seq):
        if j < len(ref_seq) and aa == ref_seq[j]:
            row.append(0)
        else:
            row.append(aa_to_idx.get(aa, 0))
    diff_matrix.append(row)

diff_matrix = np.array(diff_matrix)

cmap = ListedColormap(["#f0f0f0"] + [c for c in sns.color_palette("tab20", 20)])
bounds = np.arange(0, 22) - 0.5
norm = BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(min(40, len(ref_seq) * 0.2), max(6, num_to_plot * 0.06)))
ax = sns.heatmap(
    diff_matrix, cmap=cmap, norm=norm,
    cbar=True, linewidths=0.3, linecolor="white",
)

for start, end in fragment_bounds:
    ax.axvline(x=end + 0.5, color="red", linestyle="--", linewidth=1)

ax.set_xlabel("Amino Acid Position")
ax.set_ylabel("Sequence")
ax.set_title("Mutations Relative to Parent (color = mutant AA identity)")

cbar = ax.collections[0].colorbar
cbar.set_ticks(np.arange(0, 21))
cbar.set_ticklabels(["Same"] + aa_list)
cbar.set_label("Amino Acid")

plt.tight_layout()
plt.show()

## 9. Export FASTA

Write the selected batch to a FASTA file with well positions for plate
loading. Positive and negative controls are appended at the end.

In [ ]:
parent_name = connector.join(["1.g20", "2.g20", "3.g20", "4.g20", "5.g20"])

fasta_entries = []
for i, row in batch_df.iterrows():
    tag = row.get("selection_tag", "selected")
    name = row.get("name", f"candidate_{i}")
    fasta_entries.append((f"{name}:{tag}", row["sequence"]))

# Append controls
fasta_entries.append((parent_name + ":positive_control", ref_seq))
fasta_entries.append(("negative_control", ""))

fasta_lines = []
for i, (name, seq) in enumerate(fasta_entries):
    well = wp384[i] if i < len(wp384) else f"overflow_{i}"
    fasta_lines.append(f">{well};{name}\n{seq}\n")

# Preview
print(f"Total entries: {len(fasta_entries)} (including controls)")
print("\nFirst 5 entries:")
for line in fasta_lines[:10]:
    print(" ", line.strip()[:80])

# Uncomment to write:
# output_path = Path("../outputs/next_round_batch.fasta")
# output_path.parent.mkdir(parents=True, exist_ok=True)
# with open(output_path, "w") as f:
#     f.writelines(fasta_lines)
# print(f"\nSaved to {output_path}")

## Summary & Tips

**Selection strategy guidelines:**

| Round stage | Recommendation |
|---|---|
| Early (rounds 1–2) | Lean toward diversity; model not yet well-calibrated |
| Middle (rounds 3–4) | Mix of top-k exploitation + diversity exploration |
| Late (rounds 5+) | Mostly top-k; model calibration is improving |

**Other considerations:**
- Include a few sequences spanning a range of predicted mu to assess model
  calibration (i.e. are high-mu predictions actually better?).
- Check fragment coverage to ensure the model can learn interactions.
- If multiple gamma values produce similar top candidates, the model is
  confident about those sequences.

After experimental testing, feed the new data back into
[`model_data_preparation.ipynb`](model_data_preparation.ipynb) and retrain
the surrogate for the next round.